# 2026 FIFA World Cup — Notebook 3: Tournament Simulation
10,000 simulations | Full bracket | Group stage → Final

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
from collections import defaultdict
from itertools import combinations
from scipy.stats import poisson
warnings.filterwarnings('ignore')
np.random.seed(42)
DATA_DIR = '../data'

with open(f'{DATA_DIR}/model.pkl','rb') as f:
    saved = pickle.load(f)
clf     = saved['classifier']
pr_home = saved['poisson_home']
pr_away = saved['poisson_away']
FEATS   = saved['features']
P_FEATS = saved['poisson_features']

team_feat   = pd.read_csv(f'{DATA_DIR}/team_features.csv')
player_sum  = pd.read_csv(f'{DATA_DIR}/player_summary.csv')
dark_horses = pd.read_csv(f'{DATA_DIR}/dark_horse_scores.csv')

GROUPS = {
    'A':['Mexico','South Africa','South Korea','Czech Republic'],
    'B':['Canada','Bosnia and Herzegovina','Qatar','Switzerland'],
    'C':['Brazil','Morocco','Haiti','Scotland'],
    'D':['United States','Paraguay','Australia','Turkey'],
    'E':['Germany','Cote dIvoire','Thailand','Chile'],
    'F':['Spain','Egypt','Tunisia','Costa Rica'],
    'G':['England','Croatia','Ghana','Panama'],
    'H':['France','Senegal','Iraq','Norway'],
    'I':['Argentina','Algeria','Austria','Jordan'],
    'J':['Portugal','DR Congo','Uzbekistan','Colombia'],
    'K':['Netherlands','Iran','New Zealand','Japan'],
    'L':['Belgium','Uruguay','Sweden','Saudi Arabia'],
}
ALL_TEAMS = [t for g in GROUPS.values() for t in g]
feat_idx = team_feat.set_index('team')

def gs(team, col, default=0.0):
    try:
        v = feat_idx.loc[team, col]
        return float(v) if not pd.isna(v) else default
    except: return default

print('Model and data loaded')
print(f'Teams: {len(ALL_TEAMS)} | Groups: {len(GROUPS)}')

## 1 · Match Engine

In [ ]:
def match_probs(home, away, knockout=False):
    hr = gs(home,'rank_score',0.4)
    ar = gs(away,'rank_score',0.4)
    feat = np.array([[hr-ar,
        gs(home,'win_rate',0.33)-gs(away,'win_rate',0.33),
        gs(home,'avg_gf',1.2)-gs(away,'avg_gf',1.2),
        gs(home,'avg_ga',1.2)-gs(away,'avg_ga',1.2),
        gs(home,'avg_gd',0.0)-gs(away,'avg_gd',0.0),
        hr, ar, int(knockout),
        gs(home,'avg_xg_scored',1.0)-gs(away,'avg_xg_scored',1.0)
    ]])
    p = clf.predict_proba(feat)[0]
    ph,pd_,pa = p[0],p[1],p[2]
    rank_diff = hr - ar
    rank_ph = 1/(1+np.exp(-4*rank_diff))
    rank_pa = 1 - rank_ph
    rank_pd = 0.25
    norm = rank_ph + rank_pd + rank_pa
    rank_ph /= norm; rank_pa /= norm; rank_pd /= norm
    ph = 0.5*ph + 0.5*rank_ph
    pa = 0.5*pa + 0.5*rank_pa
    pd_ = 0.5*pd_ + 0.5*rank_pd
    if knockout:
        ph += pd_/2; pa += pd_/2; pd_ = 0
    return ph, pd_, pa

def predict_scoreline(home, away):
    hr = gs(home,'rank_score',0.4)
    ar = gs(away,'rank_score',0.4)
    xgd = gs(home,'avg_xg_scored',1.0)-gs(away,'avg_xg_scored',1.0)
    pf = np.array([[hr-ar,
        gs(home,'avg_gf',1.2)-gs(away,'avg_gf',1.2),
        gs(home,'avg_ga',1.2)-gs(away,'avg_ga',1.2),
        hr, ar, xgd]])
    af = pf.copy(); af[0,0]*=-1; af[0,5]*=-1
    lh = max(0.3, pr_home.predict(pf)[0])
    la = max(0.3, pr_away.predict(af)[0])
    best_p, best_s = 0, (1,1)
    for hg in range(8):
        for ag in range(8):
            p = poisson.pmf(hg,lh)*poisson.pmf(ag,la)
            if p > best_p: best_p=p; best_s=(hg,ag)
    return best_s, lh, la

def simulate_match(home, away, knockout=False):
    ph,pd_,pa = match_probs(home,away,knockout)
    outcome = np.random.choice(['home','draw','away'],p=[ph,pd_,pa])
    lh = max(0.3, gs(home,'avg_gf',1.2))
    la = max(0.3, gs(away,'avg_gf',1.2))
    if outcome=='home':
        while True:
            hg,ag = np.random.poisson(lh),np.random.poisson(la)
            if hg>ag: break
        winner=home
    elif outcome=='away':
        while True:
            hg,ag = np.random.poisson(lh),np.random.poisson(la)
            if ag>hg: break
        winner=away
    else:
        if knockout:
            # Penalty shootout
            hg = ag = np.random.poisson(lh)
            while hg==ag: hg,ag = np.random.poisson(lh),np.random.poisson(la)
            winner = home if np.random.random()<ph/(ph+pa) else away
        else:
            while True:
                hg,ag = np.random.poisson(lh),np.random.poisson(la)
                if hg==ag: break
            winner=None
    return hg,ag,winner

print('Match engine ready')
print('Sample predictions:')
for h,a in [('France','Germany'),('Brazil','Argentina'),('England','Spain')]:
    ph,pd_,pa = match_probs(h,a)
    sc,_,_ = predict_scoreline(h,a)
    print(f'  {h} vs {a}: {h} {ph:.0%} | Draw {pd_:.0%} | {a} {pa:.0%} | Score: {sc[0]}-{sc[1]}')

## 2 · Group Stage Simulator

In [ ]:
def simulate_group(teams):
    st = {t:{'pts':0,'gf':0,'ga':0,'gd':0,'w':0,'d':0,'l':0} for t in teams}
    h2h = {t:{o:{'pts':0,'gd':0} for o in teams if o!=t} for t in teams}
    for home,away in combinations(teams,2):
        hg,ag,_ = simulate_match(home,away)
        st[home]['gf']+=hg; st[home]['ga']+=ag; st[home]['gd']+=hg-ag
        st[away]['gf']+=ag; st[away]['ga']+=hg; st[away]['gd']+=ag-hg
        if hg>ag:
            st[home]['pts']+=3; st[home]['w']+=1; st[away]['l']+=1
            h2h[home][away]['pts']+=3; h2h[home][away]['gd']+=hg-ag
        elif hg==ag:
            st[home]['pts']+=1; st[away]['pts']+=1
            st[home]['d']+=1; st[away]['d']+=1
            h2h[home][away]['pts']+=1; h2h[away][home]['pts']+=1
        else:
            st[away]['pts']+=3; st[away]['w']+=1; st[home]['l']+=1
            h2h[away][home]['pts']+=3; h2h[away][home]['gd']+=ag-hg
    df = pd.DataFrame(st).T.reset_index().rename(columns={'index':'team'})
    return df.sort_values(['pts','gd','gf'],ascending=False).reset_index(drop=True)

print('Group simulator ready')

## 3 · Full Tournament Simulation (10,000x)

In [ ]:
N = 10000
win_counts    = defaultdict(int)
round_counts  = defaultdict(lambda: defaultdict(int))
goal_tallies  = defaultdict(list)

print(f'Running {N:,} simulations...')
for sim in range(N):
    # GROUP STAGE
    group_winners   = []
    group_runners   = []
    third_place_teams = []

    for grp, teams in GROUPS.items():
        standing = simulate_group(teams)
        group_winners.append(standing.iloc[0]['team'])
        group_runners.append(standing.iloc[1]['team'])
        third_place_teams.append({
            'team': standing.iloc[2]['team'],
            'pts':  standing.iloc[2]['pts'],
            'gd':   standing.iloc[2]['gd'],
            'gf':   standing.iloc[2]['gf'],
        })

    # Best 8 third-place teams
    third_df = pd.DataFrame(third_place_teams).sort_values(
        ['pts','gd','gf'], ascending=False
    ).head(8)
    best_thirds = third_df['team'].tolist()

    # R32: 32 teams total
    r32 = group_winners + group_runners + best_thirds
    assert len(r32)==32, f'R32 should have 32 teams, got {len(r32)}'

    # KNOCKOUT ROUNDS
    def run_round(teams, rnd):
        winners = []
        np.random.shuffle(teams)
        for i in range(0, len(teams), 2):
            h,a = teams[i], teams[i+1]
            hg,ag,w = simulate_match(h,a,knockout=True)
            if w is None:
                w = h if np.random.random()<0.5 else a
            round_counts[w][rnd]+=1
            goal_tallies[w].append(hg if w==h else ag)
            winners.append(w)
        return winners

    r16    = run_round(r32, 'r16')
    qf     = run_round(r16, 'qf')
    sf     = run_round(qf,  'sf')
    # 3rd place playoff
    third_place = run_round([t for t in qf if t not in sf], '3rd')
    final  = run_round(sf,  'final')
    champion = final[0]
    win_counts[champion]+=1

    if sim%2000==0 and sim>0:
        print(f'  {sim:,} done...')

print(f'All {N:,} simulations complete!')

## 4 · Results

In [ ]:
results = []
for team in ALL_TEAMS:
    results.append({
        'team': team,
        'group': next(g for g,ts in GROUPS.items() if team in ts),
        'fifa_rank': int(gs(team,'fifa_rank',99)),
        'p_r16':    round_counts[team]['r16']   /N,
        'p_qf':     round_counts[team]['qf']    /N,
        'p_sf':     round_counts[team]['sf']    /N,
        'p_final':  round_counts[team]['final'] /N,
        'p_winner': win_counts[team]            /N,
        'avg_ko_goals': np.mean(goal_tallies[team]) if goal_tallies[team] else 0,
    })

results_df = pd.DataFrame(results).sort_values('p_winner',ascending=False).reset_index(drop=True)
results_df['sim_rank'] = results_df.index+1

print('TOP 16 PREDICTED WIN PROBABILITIES')
print('='*60)
for _,row in results_df.head(16).iterrows():
    bar = '#'*int(row['p_winner']*200)
    print(f"  {int(row['sim_rank']):>2}. {row['team']:<26} {row['p_winner']:>5.1%}  {bar}")

## 5 · Golden Boot Predictions

In [ ]:
ROUND_MATCHES = {'r16':1,'qf':1,'sf':1,'final':1}
golden_boot = []

for _,row in results_df.iterrows():
    team = row['team']
    exp_matches = sum(row[f'p_{r}']*m for r,m in ROUND_MATCHES.items()) + 3
    tp = player_sum[player_sum['team']==team]
    if len(tp)==0: continue
    top = tp.sort_values('goals_per_match',ascending=False).iloc[0]
    golden_boot.append({
        'team':team,
        'player':top['player_name'],
        'goals_per_match':round(top['goals_per_match'],3),
        'total_wc_goals':int(top['total_goals']),
        'expected_matches':round(exp_matches,2),
        'projected_goals':round(top['goals_per_match']*exp_matches,2),
    })

gb_df = pd.DataFrame(golden_boot).sort_values('projected_goals',ascending=False).reset_index(drop=True)
print('GOLDEN BOOT PREDICTIONS')
print('='*65)
for _,row in gb_df.head(10).iterrows():
    print(f"  {row['player']:<28} ({row['team']:<14}) {row['projected_goals']:.1f} proj goals")

## 6 · Generate Predicted Bracket

In [ ]:
# Run ONE deterministic simulation for the bracket display
# Use most likely outcomes based on probabilities
bracket_matches = []

def det_match(home, away, knockout=False):
    ph,pd_,pa = match_probs(home,away,knockout)
    sc,lh,la = predict_scoreline(home,away)
    if ph>pa and ph>pd_: winner=home
    elif pa>ph and pa>pd_: winner=away
    else: winner=home if ph>=pa else away
    bracket_matches.append({
        'home':home,'away':away,
        'pred_home_score':sc[0],'pred_away_score':sc[1],
        'home_win_prob':round(ph,3),'draw_prob':round(pd_,3),'away_win_prob':round(pa,3),
        'predicted_winner':winner,'knockout':knockout
    })
    return winner

# Group stage
det_winners = []; det_runners = []; det_thirds = []
for grp,teams in GROUPS.items():
    st = {t:{'pts':0,'gf':0,'ga':0,'gd':0} for t in teams}
    for home,away in combinations(teams,2):
        ph,pd_,pa = match_probs(home,away)
        sc,_,_ = predict_scoreline(home,away)
        hg,ag = sc
        if ph>pa and ph>pd_: hg=max(hg,1); ag=max(0,hg-1)
        elif pa>ph and pa>pd_: ag=max(ag,1); hg=max(0,ag-1)
        else: hg=ag=1
        st[home]['gf']+=hg; st[home]['ga']+=ag; st[home]['gd']+=hg-ag
        st[away]['gf']+=ag; st[away]['ga']+=hg; st[away]['gd']+=ag-hg
        if hg>ag: st[home]['pts']+=3
        elif hg==ag: st[home]['pts']+=1; st[away]['pts']+=1
        else: st[away]['pts']+=3
        bracket_matches.append({'round':'Group '+grp,'home':home,'away':away,
            'pred_home_score':hg,'pred_away_score':ag,
            'home_win_prob':round(ph,3),'draw_prob':round(pd_,3),'away_win_prob':round(pa,3),
            'predicted_winner':home if hg>ag else (away if ag>hg else 'Draw'),
            'knockout':False})
    ranking = sorted(st.keys(), key=lambda t:(-st[t]['pts'],-st[t]['gd'],-st[t]['gf']))
    det_winners.append(ranking[0])
    det_runners.append(ranking[1])
    det_thirds.append({'team':ranking[2],'pts':st[ranking[2]]['pts'],
                       'gd':st[ranking[2]]['gd'],'gf':st[ranking[2]]['gf']})

third_sorted = sorted(det_thirds, key=lambda x:(-x['pts'],-x['gd'],-x['gf']))
best8thirds = [t['team'] for t in third_sorted[:8]]
r32_teams = det_winners+det_runners+best8thirds

# Knockout
r32_copy = r32_teams.copy()
np.random.seed(0)
np.random.shuffle(r32_copy)

def run_det_round(teams, rnd):
    winners = []
    for i in range(0,len(teams),2):
        h,a = teams[i],teams[i+1]
        w = det_match(h,a,knockout=True)
        bm = bracket_matches[-1].copy()
        bm['round'] = rnd
        bracket_matches[-1] = bm
        winners.append(w)
    return winners

r16_w = run_det_round(r32_copy,'R32')
qf_w  = run_det_round(r16_w,'R16')
sf_w  = run_det_round(qf_w,'QF')
sf_losers = [t for t in qf_w if t not in sf_w]
third_w = run_det_round(sf_losers,'3rd Place')
final_w = run_det_round(sf_w,'Final')
champion = final_w[0]

bracket_df = pd.DataFrame(bracket_matches)
print(f'Predicted champion: {champion}')
print(f'Bracket matches: {len(bracket_df)}')

## 7 · Save All Outputs

In [ ]:
results_df.to_csv(f'{DATA_DIR}/simulation_results.csv', index=False)
gb_df.to_csv(f'{DATA_DIR}/golden_boot.csv', index=False)
bracket_df.to_csv(f'{DATA_DIR}/bracket_predictions.csv', index=False)

# Save fixtures with predicted scores
import os
fixtures = []
for grp,teams in GROUPS.items():
    for home,away in combinations(teams,2):
        sc,_,_ = predict_scoreline(home,away)
        ph,pd_,pa = match_probs(home,away)
        fixtures.append({
            'group':grp,'home':home,'away':away,
            'pred_home':sc[0],'pred_away':sc[1],
            'home_win_prob':round(ph,3),
            'draw_prob':round(pd_,3),
            'away_win_prob':round(pa,3),
            'actual_home_score':None,'actual_away_score':None,'played':False
        })
pd.DataFrame(fixtures).to_csv(f'{DATA_DIR}/fixtures_2026.csv', index=False)

print('Saved:')
print(f'  simulation_results.csv   {results_df.shape}')
print(f'  golden_boot.csv          {gb_df.shape}')
print(f'  bracket_predictions.csv  {bracket_df.shape}')
print(f'  fixtures_2026.csv        {len(fixtures)} group matches')
print(f'Predicted Winner: {champion}')
print('Notebook 3 complete!')